# WebFlor · sólo el complemento necesario

**Entra:** la descarga local original, sin conectar a SQL por defecto.
**Salen:** `proyecciones_webflor_limpio.parquet` y `coeficientes_webflor_limpio.parquet`.

Las cinco fuentes Excel son oficiales para producción, planos, clima, pilotos y benchmark del ingeniero. SQL aporta dos hechos distintos: **lo que WebFlor almacenó como proyección** y **sus coeficientes internos**, que no se pueden recuperar sustituyéndolos por el estimado manual o por una curva piloto.

Se necesitan 16 tablas de origen porque varias son dependencias pequeñas de identidad; no son 16 nuevas bases analíticas. No se ejecuta ningún SP ni proceso de generación.

## 1. Imports y rutas

In [1]:
from pathlib import Path
import sys
import re
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "Datos").is_dir() and (p / "Notebooks").is_dir())
from datetime import date
import unicodedata
from pathlib import PurePosixPath
from zipfile import ZipFile
from lxml import etree

def normalizar(serie):
    return (serie.astype("string").str.normalize("NFKD")
            .str.replace("[\u0300-\u036f]", "", regex=True)
            .str.replace(r"\s+", " ", regex=True).str.strip().str.upper()
            .replace({"": pd.NA, "NAN": pd.NA, "NONE": pd.NA, "<NA>": pd.NA}))

def semana_operativa(anio, semana):
    """Lunes representativo. Toda etiqueta 53 se convierte en 01 del año siguiente."""
    y = pd.to_numeric(anio, errors="coerce").astype("Int64")
    w = pd.to_numeric(semana, errors="coerce").astype("Int64")
    y = y + w.eq(53).fillna(False).astype(int)
    w = w.mask(w.eq(53), 1)
    etiquetas = y.astype("string") + "-" + w.astype("string")
    conversion = {}
    for etiqueta in etiquetas.dropna().unique():
        try:
            yy, ww = map(int, etiqueta.split("-"))
            conversion[etiqueta] = pd.Timestamp(date.fromisocalendar(yy, ww, 1)) if 1 <= ww <= 52 else pd.NaT
        except (ValueError, OverflowError):
            conversion[etiqueta] = pd.NaT
    return pd.to_datetime(etiquetas.map(conversion))

SALIDA = ROOT / "Datos_analiticos_proyecto"
SALIDA.mkdir(exist_ok=True)
pd.set_option("display.max_columns", 35)
print("Fuentes originales:", ROOT / "Datos")
print("Salida oficial:", SALIDA)

SQL = ROOT / "Notebooks/Extracción SQL/Datos"
ACTUALIZAR_SQL = False  # dejar False para reconstruir sólo con archivos locales
ESPECIFICACIONES = [['01_maestros_y_geografia/Basico_Finca.csv', 'Basico', 'Finca', 'IdFinca,NomFinca', 'Finca; maestro actual'], ['01_maestros_y_geografia/Mapa_Invernadero.csv', 'Mapa', 'Invernadero', 'IdInvernadero,IdFinca,NomInvernadero', 'Invernadero → finca'], ['01_maestros_y_geografia/Mapa_CamaFisica.csv', 'Mapa', 'CamaFisica', 'IdCamaFisica,IdInvernadero', 'Cama → invernadero'], ['01_maestros_y_geografia/TablasBasicas_Variedad.csv', 'TablasBasicas', 'Variedad', 'IdVariedad,IdProducto,IdColor,NomVariedad', 'Código de variedad; maestro actual'], ['01_maestros_y_geografia/TablasBasicas_Producto.csv', 'TablasBasicas', 'Producto', 'IdProducto,NomProducto', 'Producto'], ['01_maestros_y_geografia/TablasBasicas_Color.csv', 'TablasBasicas', 'Color', 'IdColor,NomColor', 'Color'], ['01_maestros_y_geografia/TablasBasicas_Ciclo.csv', 'TablasBasicas', 'Ciclo', 'IdCiclo,NomCiclo,PickPeriodo,PickForma,PickTipoProyeccion', 'Unidad y método del coeficiente'], ['01_maestros_y_geografia/TablasBasicas_CicloVariedad.csv', 'TablasBasicas', 'CicloVariedad', 'IdCicloVariedad,IdFinca,IdVariedad,IdCiclo', 'Configuración finca/variedad/ciclo'], ['02_siembra_y_erradicacion/Siembra_SiembraCama.csv', 'Siembra', 'SiembraCama', 'IdSiembra,IdVariedad,IdCamaFisica,IdCicloVariedad,FechaSiembra,FechaAuditoria', 'Identidad de siembra actual'], ['02_siembra_y_erradicacion/Siembra_SiembraHistorico.csv', 'Siembra', 'SiembraHistorico', 'IdSiembraHistorico,IdSiembra,IdVariedad,IdCamaFisica,IdCicloVariedad,FechaSiembra,FechaAuditoria,FechaErradicacion', 'Identidad en evento archivado'], ['02_siembra_y_erradicacion/Siembra_SiembraCamaLog.csv', 'Siembra', 'SiembraCamaLog', 'IdSiembraCamaLog,IdSiembra,IdVariedad,IdCamaFisica,IdCicloVariedad,FechaSiembra,FechaAuditoria,EsErradicacion', 'Identidad en erradicación parcial'], ['03_productividad/Siembra_CurvaCiclo.csv', 'Siembra', 'CurvaCiclo', 'IdCurvaCiclo,NomCurva,IdCicloVariedad', 'Curva → configuración'], ['03_productividad/Siembra_CurvaCoeficiente.csv', 'Siembra', 'CurvaCoeficiente', 'IdCurvaCoeficiente,IdCurvaCiclo,SemanaDia,Coeficiente,Estado,FechaAuditoria', 'Coeficiente interno por edad; no curva piloto'], ['04_proyecciones/Siembra_ProyeccionSemana.csv', 'Siembra', 'ProyeccionSemana', 'IdProyeccionSemana,IdSiembra,AnioProyeccion,SemanaProyeccion,ProyeccionFinal,ProyeccionInicial,Coeficiente,FechaAuditoria', 'Estado semanal almacenado'], ['04_proyecciones/Siembra_HistoricoProyeccionSemana.csv', 'Siembra', 'HistoricoProyeccionSemana', 'IdHistoricoProyeccionSemana,IdProyeccionSemana,IdSiembra,AnioProyeccion,SemanaProyeccion,ProyeccionFinal,ProyeccionInicial,Coeficiente,FechaAuditoria', 'Archivo semanal; auditoría = archivo, no emisión'], ['04_proyecciones/Siembra_HistoricoProyeccionDia.csv', 'Siembra', 'HistoricoProyeccionDia', 'IdHistoricoProyeccionDia,IdProyeccionDia,IdSiembra,AnioProyeccion,SemanaProyeccion,FechaProyeccion,ProyeccionFinal,ProyeccionInicial,Coeficiente,FechaAuditoria', 'Archivo diario; no sumar a la representación semanal']]
display(pd.DataFrame(ESPECIFICACIONES,columns=["archivo","esquema","objeto","columnas","aporte"]))


Fuentes originales: C:\Materias y Cosas Técnicas Uniandes\Proyecto de grado\Proyecto de grado 2\Datos
Salida oficial: C:\Materias y Cosas Técnicas Uniandes\Proyecto de grado\Proyecto de grado 2\Datos_analiticos_proyecto


,archivo,esquema,objeto,columnas,aporte
0,01_maestros_y_geografia/Basico_Finca.csv,Basico,Finca,"IdFinca,NomFinca",Finca; maestro actual
1,01_maestros_y_geografia/Mapa_Invernadero.csv,Mapa,Invernadero,"IdInvernadero,IdFinca,NomInvernadero",Invernadero → finca
2,01_maestros_y_geografia/Mapa_CamaFisica.csv,Mapa,CamaFisica,"IdCamaFisica,IdInvernadero",Cama → invernadero
3,01_maestros_y_geografia/TablasBasicas_Variedad...,TablasBasicas,Variedad,"IdVariedad,IdProducto,IdColor,NomVariedad",Código de variedad; maestro actual
4,01_maestros_y_geografia/TablasBasicas_Producto...,TablasBasicas,Producto,"IdProducto,NomProducto",Producto
5,01_maestros_y_geografia/TablasBasicas_Color.csv,TablasBasicas,Color,"IdColor,NomColor",Color
6,01_maestros_y_geografia/TablasBasicas_Ciclo.csv,TablasBasicas,Ciclo,"IdCiclo,NomCiclo,PickPeriodo,PickForma,PickTip...",Unidad y método del coeficiente
7,01_maestros_y_geografia/TablasBasicas_CicloVar...,TablasBasicas,CicloVariedad,"IdCicloVariedad,IdFinca,IdVariedad,IdCiclo",Configuración finca/variedad/ciclo
8,02_siembra_y_erradicacion/Siembra_SiembraCama.csv,Siembra,SiembraCama,"IdSiembra,IdVariedad,IdCamaFisica,IdCicloVarie...",Identidad de siembra actual
9,02_siembra_y_erradicacion/Siembra_SiembraHisto...,Siembra,SiembraHistorico,"IdSiembraHistorico,IdSiembra,IdVariedad,IdCama...",Identidad en evento archivado


## 2. Decisión conceptual sobre las extracciones anteriores

| Familia | Decisión | Grano / aporte | Comparación con Excel y consumidor |
|---|---|---|---|
| Proyección semanal actual e histórica | CONSERVAR | Siembra/año/semana; estado o archivo | No equivale al ingeniero. Limpieza SQL y EDA retrospectivo. |
| Coeficientes/curvas/ciclos | CONSERVAR | Curva/edad con unidad definida por ciclo | No equivalen a curva empírica. Auxiliar de interpretación; sin vigencia histórica certificada. |
| SiembraCama, Histórico y Log | CONSERVAR | Siembra o evento | Necesarios para identificar proyecciones archivadas, no para duplicar los planos. |
| Finca, invernadero, cama, variedad, producto, color | CONSERVAR | Una entidad por ID | Dependencias para relacionar los hechos anteriores. |
| Proyección diaria histórica | CONSERVAR | Siembra/fecha objetivo | Archivo diario separado por periodicidad; no sumar con semanal. |
| CorteVariedad / informe de producción diaria | DESCARTAR del pipeline | Corte o reporte diario | Producción oficial = Excel; no sumar ambas. El informe ya se contrastó con la entrega anterior. Detalle SQL distinto no se declara idéntico sin prueba. |
| Clasificación y causas | DESCARTAR | Evento de clasificación/causa | Podría servir para validar calidad/exportable; no es equivalente automático a corte. |
| Respaldos SQL de corte fechados | DESCARTAR | Estado de corte en fecha pasada | No son versiones técnicas creadas por nuestros notebooks; El usuario autorizó eliminar las descargas descartadas, aun sin garantizar reconstruir su estado previo. |
| ProyeccionSemanaView, agregados finca/invernadero/grado | DESCARTAR del pipeline | Representaciones agregadas del mismo proceso | No duplicar cantidades. Se conserva el hecho semanal detallado. |
| Actividades y mano de obra | DESCARTAR del pipeline | Movimientos descargados vacíos; maestros no miden horas | Sin variable laboral defendible; no usar datos personales del empleado como sustituto. |
| Sanidad/MIP | DESCARTAR del pipeline | Descargas vacías | Sin información adicional utilizable. |
| Inventario poscosecha | DESCARTAR del pipeline | Existencias/movimientos de inventario | No es producción real; fuera de la pregunta actual. |
| Parcial timeout | DESCARTAR | Extracción incompleta | No debe alimentar ningún análisis. |

El usuario autorizó la eliminación definitiva de las descargas descartadas y las exportaciones derivadas. Se conservan los 16 archivos fuente necesarios y la captura original `ultima_proyeccion_existente.csv`: al contrastarla, sus registros desde 2021 ya estaban representados; sólo aporta 44 registros de GAITANA PPTO de 2020. Estos se conservan identificados, fuera del panel de producción real. No se generan nuevas copias ni archivos intermedios.


## 3. Extracción opcional, exclusivamente SELECT

Con `ACTUALIZAR_SQL=False` no hay conexión. Una actualización autorizada lee los 16 objetos en memoria y genera solamente las dos tablas finales de la sección 8. No escribe un CSV por tabla, dependencia o join. Los archivos locales conservados son fuentes originales de la investigación, no exportaciones nuevas. No ejecutar procedimientos.


In [2]:
tablas = {}
if ACTUALIZAR_SQL:
    import os
    import pyodbc
    requeridas = ["WEBFLOR_SERVER","WEBFLOR_DATABASE","WEBFLOR_USER","WEBFLOR_PASSWORD"]
    assert all(os.environ.get(k) for k in requeridas),"Configurar conexión mediante variables de entorno"
    conn_str = ("DRIVER={ODBC Driver 18 for SQL Server};"
                f"SERVER={os.environ['WEBFLOR_SERVER']};DATABASE={os.environ['WEBFLOR_DATABASE']};"
                f"UID={os.environ['WEBFLOR_USER']};PWD={os.environ['WEBFLOR_PASSWORD']};"
                "Encrypt=yes;TrustServerCertificate=yes;ApplicationIntent=ReadOnly;")
    conn = pyodbc.connect(conn_str,timeout=15)
    try:
        for relativo,esquema,objeto,columnas,aporte in ESPECIFICACIONES:
            campos = ",".join(f"[{c}]" for c in columnas.split(","))
            filtro = " WHERE AnioProyeccion >= 2021" if objeto in ["ProyeccionSemana","HistoricoProyeccionSemana","HistoricoProyeccionDia"] else ""
            consulta = f"SELECT {campos} FROM [{esquema}].[{objeto}]{filtro}"
            tablas[esquema+"."+objeto] = pd.read_sql_query(consulta,conn)
            print(esquema+"."+objeto,len(tablas[esquema+"."+objeto]))
    finally:
        conn.close()
else:
    print("Modo local. No se realizó ninguna conexión SQL.")


Modo local. No se realizó ninguna conexión SQL.


## 4. Carga local y cobertura

Se leen únicamente columnas necesarias. Cada tabla se muestra con filas, primera llave candidata y rango temporal realmente presente; una fecha de auditoría de maestro no es vigencia histórica.

In [3]:
inventario_sql = []
for relativo,esquema,objeto,columnas,aporte in ESPECIFICACIONES:
    df = tablas[esquema+"."+objeto] if ACTUALIZAR_SQL else pd.read_csv(SQL/relativo,usecols=columnas.split(","),low_memory=False)
    tablas[esquema+"."+objeto] = df
    fecha_col = "AnioProyeccion" if "AnioProyeccion" in df else "FechaAuditoria" if "FechaAuditoria" in df else None
    inventario_sql.append({"objeto":esquema+"."+objeto,"filas":len(df),"columnas":len(df.columns),
        "llave_candidata":df.columns[0],"duplicados_llave":int(df.duplicated(df.columns[0]).sum()),
        "campo_cobertura":fecha_col,"desde":str(df[fecha_col].min()) if fecha_col else "maestro sin vigencia",
        "hasta":str(df[fecha_col].max()) if fecha_col else "maestro sin vigencia","aporte":aporte})
display(pd.DataFrame(inventario_sql))


,objeto,filas,columnas,llave_candidata,duplicados_llave,campo_cobertura,desde,hasta,aporte
0,Basico.Finca,78,2,IdFinca,0,NaN,maestro sin vigencia,maestro sin vigencia,Finca; maestro actual
1,Mapa.Invernadero,90,3,IdInvernadero,0,NaN,maestro sin vigencia,maestro sin vigencia,Invernadero → finca
2,Mapa.CamaFisica,17084,2,IdCamaFisica,0,NaN,maestro sin vigencia,maestro sin vigencia,Cama → invernadero
3,TablasBasicas.Variedad,2974,4,IdVariedad,0,NaN,maestro sin vigencia,maestro sin vigencia,Código de variedad; maestro actual
4,TablasBasicas.Producto,138,2,IdProducto,0,NaN,maestro sin vigencia,maestro sin vigencia,Producto
5,TablasBasicas.Color,54,2,IdColor,0,NaN,maestro sin vigencia,maestro sin vigencia,Color
6,TablasBasicas.Ciclo,27,5,IdCiclo,0,NaN,maestro sin vigencia,maestro sin vigencia,Unidad y método del coeficiente
7,TablasBasicas.CicloVariedad,2800,4,IdCicloVariedad,0,NaN,maestro sin vigencia,maestro sin vigencia,Configuración finca/variedad/ciclo
8,Siembra.SiembraCama,11786,6,IdSiembra,0,FechaAuditoria,2020-10-20 10:08:52.290,2026-09-22 08:58:18.407,Identidad de siembra actual
9,Siembra.SiembraHistorico,29066,8,IdSiembraHistorico,0,FechaAuditoria,2023-09-19 06:32:45.887,2026-09-22 08:57:41.297,Identidad en evento archivado


## 5. Identidad de siembra: primero llaves, después cantidades

Cama → invernadero → finca y ciclo-variedad → finca deben concordar. La identidad actual sirve para el estado actual; no se arrastra indiscriminadamente hacia archivos anteriores. Los maestros describen la clasificación descargada, no una historia certificada de nombres.

In [4]:
cama = tablas["Mapa.CamaFisica"]
inv = tablas["Mapa.Invernadero"][["IdInvernadero","IdFinca"]]
ciclo_finca = tablas["TablasBasicas.CicloVariedad"][["IdCicloVariedad","IdFinca"]].rename(columns={"IdFinca":"finca_ciclo"})
identidades = []
for obj in ["Siembra.SiembraCama","Siembra.SiembraHistorico","Siembra.SiembraCamaLog"]:
    df = tablas[obj].copy()
    df["fuente_identidad"] = obj
    df["evento_archivo"] = obj=="Siembra.SiembraHistorico"
    if obj.endswith("Log"):
        df["evento_archivo"] = df.EsErradicacion.astype("string").str.lower().isin(["1","1.0","true"])
    df = df.merge(cama,on="IdCamaFisica",how="left",validate="many_to_one")
    df = df.merge(inv,on="IdInvernadero",how="left",validate="many_to_one")
    df = df.merge(ciclo_finca,on="IdCicloVariedad",how="left",validate="many_to_one")
    conflicto = df.IdFinca.notna() & df.finca_ciclo.notna() & df.IdFinca.ne(df.finca_ciclo)
    df["IdFinca"] = df.IdFinca.fillna(df.finca_ciclo).where(~conflicto)
    df["fecha_evento"] = pd.to_datetime(df.FechaAuditoria,errors="coerce",format="mixed")
    identidades.append(df[["IdSiembra","IdVariedad","IdFinca","fuente_identidad","evento_archivo","fecha_evento"]])
identidades = pd.concat(identidades,ignore_index=True)
actual_id = identidades.loc[identidades.fuente_identidad.eq("Siembra.SiembraCama"),["IdSiembra","IdVariedad","IdFinca"]]
assert not actual_id.duplicated("IdSiembra").any()
unicas = identidades[["IdSiembra","IdVariedad","IdFinca"]].drop_duplicates()
consistentes = unicas.loc[~unicas.duplicated("IdSiembra",keep=False)]
print("Siembras con identidad consistente:",len(consistentes))


Siembras con identidad consistente: 68285


## 6. Archivo de proyecciones y estado descargado

El archivo histórico se relaciona con un evento único de la misma siembra en ±30 segundos, siguiendo la evidencia SQL ya documentada. Es un enlace temporal comprobable, **no fecha de generación**. Si no es único, sólo se admite identidad consistente; no se escoge el candidato más parecido.

In [5]:
actual = tablas["Siembra.ProyeccionSemana"].copy()
historico = tablas["Siembra.HistoricoProyeccionSemana"].copy()
actual["fuente_estado"],historico["fuente_estado"] = "actual","archivado"
actual["id_registro_fuente"] = actual.IdProyeccionSemana
historico["id_registro_fuente"] = historico.IdHistoricoProyeccionSemana
diario = tablas["Siembra.HistoricoProyeccionDia"].copy()
diario["fuente_estado"] = "archivado_diario"
diario["id_registro_fuente"] = diario.IdHistoricoProyeccionDia
actual["periodicidad"],historico["periodicidad"],diario["periodicidad"] = "semana","semana","dia"
proy = pd.concat([actual,historico,diario],ignore_index=True,sort=False)
proy["fecha_auditoria"] = pd.to_datetime(proy.FechaAuditoria,errors="coerce",format="mixed")
eventos = {k:d for k,d in identidades.loc[identidades.evento_archivo].groupby("IdSiembra")}
enlaces = []
for fila in proy.loc[proy.fuente_estado.str.startswith("archivado"),["IdSiembra","fecha_auditoria"]].drop_duplicates().itertuples(index=False):
    ev = eventos.get(fila.IdSiembra)
    candidatos = ev.loc[(ev.fecha_evento-fila.fecha_auditoria).abs().le(pd.Timedelta(seconds=30))] if ev is not None else pd.DataFrame()
    enlaces.append({"IdSiembra":fila.IdSiembra,"fecha_auditoria":fila.fecha_auditoria,"n_candidatos":len(candidatos),
        "variedad_evento":candidatos.iloc[0].IdVariedad if len(candidatos)==1 else np.nan,
        "finca_evento":candidatos.iloc[0].IdFinca if len(candidatos)==1 else np.nan})
enlaces = pd.DataFrame(enlaces)
proy = proy.merge(consistentes,on="IdSiembra",how="left",validate="many_to_one")
proy = proy.merge(actual_id.rename(columns={"IdVariedad":"variedad_actual","IdFinca":"finca_actual"}),on="IdSiembra",how="left",validate="many_to_one")
proy = proy.merge(enlaces,on=["IdSiembra","fecha_auditoria"],how="left",validate="many_to_one")
es_actual = proy.fuente_estado.eq("actual")
enlace_unico = proy.fuente_estado.str.startswith("archivado") & proy.n_candidatos.eq(1)
proy.loc[es_actual,["IdVariedad","IdFinca"]] = proy.loc[es_actual,["variedad_actual","finca_actual"]].to_numpy()
proy.loc[enlace_unico,["IdVariedad","IdFinca"]] = proy.loc[enlace_unico,["variedad_evento","finca_evento"]].to_numpy()
proy["metodo_identidad"] = np.select([es_actual,enlace_unico],["siembra_actual","evento_unico_30s"],default="identidad_consistente")
for tabla,llave in [("TablasBasicas.Variedad","IdVariedad"),("TablasBasicas.Producto","IdProducto"),
                    ("TablasBasicas.Color","IdColor"),("Basico.Finca","IdFinca")]:
    proy = proy.merge(tablas[tabla],on=llave,how="left",validate="many_to_one")
for nombre,col in [("finca","NomFinca"),("producto","NomProducto"),("color","NomColor"),("variedad","NomVariedad")]:
    proy[nombre] = normalizar(proy[col])
proy["semana_inicio"] = semana_operativa(proy.AnioProyeccion,proy.SemanaProyeccion)
proy["fecha_objetivo"] = pd.to_datetime(proy.FechaProyeccion,errors="coerce",format="mixed").where(proy.periodicidad.eq("dia"),proy.semana_inicio)
# Captura original distinta: conservar sólo registros no representados en el archivo/estado.
# No se le atribuye una fecha de generación ni se suma al estado actual.
previa = pd.read_csv(SQL.parent / "ultima_proyeccion_existente.csv", low_memory=False)
previa["id_registro_fuente"] = np.arange(1, len(previa)+1)
contenido = ["IdSiembra","AnioProyeccion","SemanaProyeccion","ProyeccionInicial","ProyeccionFinal","Coeficiente"]
previa = previa.merge(proy.loc[proy.periodicidad.eq("semana"),contenido].drop_duplicates(),
                      on=contenido,how="left",validate="many_to_one",indicator=True)
previa = previa.loc[previa._merge.eq("left_only")].copy()
for nombre,col in [("finca","NomFinca"),("producto","NomProducto"),("color","nomcolor"),("variedad","NomVariedad")]:
    previa[nombre] = normalizar(previa[col])
previa["fuente_estado"],previa["periodicidad"] = "captura_previa_sin_origen","semana"
previa["metodo_identidad"] = "dimensiones_del_informe_original"
previa["semana_inicio"] = semana_operativa(previa.AnioProyeccion,previa.SemanaProyeccion)
previa["fecha_objetivo"] = previa.semana_inicio
print("Registros adicionales de captura original:", len(previa))
display(previa.groupby(["finca","AnioProyeccion"],dropna=False).size().to_frame("filas"))
proy = pd.concat([proy,previa.drop(columns="_merge")],ignore_index=True,sort=False)
fisica = ["IdSiembra","periodicidad","AnioProyeccion","SemanaProyeccion","fecha_objetivo"]
variantes = proy.groupby(fisica,dropna=False)[["ProyeccionInicial","ProyeccionFinal","Coeficiente"]].nunique().gt(1).any(axis=1).rename("conflicto_cantidad")
proy = proy.merge(variantes,on=fisica,how="left",validate="many_to_one")
proy["fecha_generacion_confirmada"] = pd.NaT
proy["admitida_predictor_historico"] = False
proy["unidad"] = "tallos proyectados WebFlor; no observado"
assert len(proy)==len(actual)+len(historico)+len(diario)+len(previa)
assert not proy.duplicated(["fuente_estado","id_registro_fuente"]).any()
display(proy.groupby(["fuente_estado","finca"],dropna=False).agg(filas=("IdSiembra","size"),
         desde=("semana_inicio","min"),hasta=("semana_inicio","max"),conflictos=("conflicto_cantidad","sum")))
print("Identidades sin resolver:",int(proy.IdFinca.isna().sum()))
display(enlaces.n_candidatos.value_counts())


Registros adicionales de captura original: 44


,,filas
finca,AnioProyeccion,
GAITANA PPTO,2020,44


filas      desde      hasta  \
fuente_estado             finca                                        
actual                    ARABELLA       98604 2025-03-31 2026-12-21   
                          GAITANA       209845 2023-07-24 2026-12-21   
archivado                 ARABELLA      476760 2022-06-27 2026-12-21   
                          GAITANA       877796 2021-01-04 2026-12-21   
archivado_diario          GAITANA        22985 2022-12-26 2023-03-13   
captura_previa_sin_origen GAITANA PPTO      44 2020-10-12 2020-12-21   

                                        conflictos  
fuente_estado             finca                     
actual                    ARABELLA               0  
                          GAITANA                0  
archivado                 ARABELLA               0  
                          GAITANA               16  
archivado_diario          GAITANA                0  
captura_previa_sin_origen GAITANA PPTO           0

Identidades sin resolver: 0


n_candidatos
1    25569
Name: count, dtype: int64

## 7. Coeficientes internos: mantener su semántica

CurvaCoeficiente.SemanaDia es edad, no semana calendario. PickPeriodo 8 indica semanas; 7 días. PickForma 44 usa área; otro valor usa plantas. Son parámetros actuales descargados, no una curva empírica ni parámetros históricos válidos desde 2021.

In [6]:
coef = tablas["Siembra.CurvaCoeficiente"].copy()
coef = coef.merge(tablas["Siembra.CurvaCiclo"],on="IdCurvaCiclo",how="left",validate="many_to_one")
coef = coef.merge(tablas["TablasBasicas.CicloVariedad"],on="IdCicloVariedad",how="left",validate="many_to_one")
coef = coef.merge(tablas["TablasBasicas.Ciclo"],on="IdCiclo",how="left",validate="many_to_one")
for tabla,llave in [("TablasBasicas.Variedad","IdVariedad"),("TablasBasicas.Producto","IdProducto"),
                    ("TablasBasicas.Color","IdColor"),("Basico.Finca","IdFinca")]:
    coef = coef.merge(tablas[tabla],on=llave,how="left",validate="many_to_one")
for nombre,col in [("finca","NomFinca"),("producto","NomProducto"),("color","NomColor"),("variedad","NomVariedad")]:
    coef[nombre] = normalizar(coef[col])
coef["unidad_edad"] = coef.PickPeriodo.map({7:"dias",8:"semanas"}).fillna("por_validar")
coef["base_formula"] = np.where(coef.PickForma.eq(44),"area","plantas")
coef["vigencia_historica_confirmada"] = False
assert len(coef)==len(tablas["Siembra.CurvaCoeficiente"])
display(coef.groupby(["finca","unidad_edad","base_formula"],dropna=False).size().to_frame("coeficientes"))


coeficientes
finca         unidad_edad base_formula              
ARABELLA      semanas     plantas               7930
ARABELLA PPTO semanas     plantas               5911
GAITANA       dias        plantas                  1
              semanas     plantas              60979
GAITANA PPTO  semanas     plantas              53746

## 8. Sólo dos salidas limpias

Se retienen identificadores, dimensiones, cantidades y advertencias. No se exportan identidades, eventos, puentes ni cada join intermedio. Los conflictos se preservan dentro de la tabla, no se ocultan en un archivo separado.

In [7]:
columnas = ["fuente_estado","periodicidad","fecha_objetivo","id_registro_fuente","IdProyeccionSemana","IdSiembra","IdVariedad","IdFinca",
            "finca","producto","color","variedad","AnioProyeccion","SemanaProyeccion","semana_inicio",
            "ProyeccionInicial","ProyeccionFinal","Coeficiente","fecha_auditoria","fecha_generacion_confirmada",
            "metodo_identidad","conflicto_cantidad","admitida_predictor_historico","unidad"]
proy[columnas].to_parquet(SALIDA/"proyecciones_webflor_limpio.parquet",index=False)
coef.to_parquet(SALIDA/"coeficientes_webflor_limpio.parquet",index=False)
print("EXPORTADOS: proyecciones_webflor_limpio.parquet",len(proy),"filas")
print("EXPORTADOS: coeficientes_webflor_limpio.parquet",len(coef),"filas")
display(proy[columnas].head())


EXPORTADOS: proyecciones_webflor_limpio.parquet 1686034 filas
EXPORTADOS: coeficientes_webflor_limpio.parquet 128567 filas


,fuente_estado,periodicidad,fecha_objetivo,id_registro_fuente,IdProyeccionSemana,IdSiembra,IdVariedad,IdFinca,finca,producto,color,variedad,AnioProyeccion,SemanaProyeccion,semana_inicio,ProyeccionInicial,ProyeccionFinal,Coeficiente,fecha_auditoria,fecha_generacion_confirmada,metodo_identidad,conflicto_cantidad,admitida_predictor_historico,unidad
0,actual,semana,2023-08-14,10504742,10504742.0,79796,1147.0,3.0,GAITANA,VERONICA,LIGHT PINK,SKYLER PINK,2023,33,2023-08-14,0,0,0.0,2023-07-17 09:57:15.123,NaT,siembra_actual,False,False,tallos proyectados WebFlor; no observado
1,actual,semana,2023-08-07,10504743,10504743.0,79796,1147.0,3.0,GAITANA,VERONICA,LIGHT PINK,SKYLER PINK,2023,32,2023-08-07,0,0,0.0,2023-07-17 09:57:15.123,NaT,siembra_actual,False,False,tallos proyectados WebFlor; no observado
2,actual,semana,2023-08-21,10504744,10504744.0,79796,1147.0,3.0,GAITANA,VERONICA,LIGHT PINK,SKYLER PINK,2023,34,2023-08-21,0,0,0.0,2023-07-17 09:57:15.123,NaT,siembra_actual,False,False,tallos proyectados WebFlor; no observado
3,actual,semana,2023-08-28,10504745,10504745.0,79796,1147.0,3.0,GAITANA,VERONICA,LIGHT PINK,SKYLER PINK,2023,35,2023-08-28,0,0,0.0,2023-07-17 09:57:15.123,NaT,siembra_actual,False,False,tallos proyectados WebFlor; no observado
4,actual,semana,2023-09-04,10504746,10504746.0,79796,1147.0,3.0,GAITANA,VERONICA,LIGHT PINK,SKYLER PINK,2023,36,2023-09-04,0,0,0.0,2023-07-17 09:57:15.123,NaT,siembra_actual,False,False,tallos proyectados WebFlor; no observado
